# Sizing / SL / PT / Kelly experiments (без переобучения моделей)

Цель: ответить на два вопроса пользователя:

1. **Какой размер позиции дефолтный?** Sweep `position_size_fraction` ∈
   {5%, 10%, 15%, 20%, 25%, 30%} и сравнение с legacy `equal_split`.
2. **Stop loss / profit target / Kelly sizing — выгодно ли?** Эксперименты
   на одних и тех же чекпоинтах модели (никаких изменений в обучении).

## Почему отдельный ноутбук

Загружаем сохранённые Primary + Meta чекпоинты из `lightgbm_meta_ensemble`,
кэшируем их test-predictions в parquet и крутим backtest с разными
:class:`TradingConfig`. Один цикл подгонки занимает ~30 мин обучения; здесь
мы запускаем десятки backtest'ов за минуты.

## Что меняется
- `cfg.trading.sizing_mode` ∈ `equal_split` | `fixed_frac` | `signal_kelly`
- `cfg.trading.position_size_fraction` ∈ {0.05, 0.10, 0.15, 0.20, 0.30}
- `cfg.trading.stop_loss_pct`, `cfg.trading.profit_target_pct`
- Для Kelly: `signal_kelly_size()` → колонка `size_fraction` в сигналах

## Что НЕ меняется
- Веса моделей (Primary ensemble + Meta) — берём с диска
- Joint thresholds (T_prim, T_meta_abs) — рассчитанные на val


## 0. Окружение

In [ ]:
from __future__ import annotations

import os, subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_OWNER = 'Pozdnyakof'
REPO_NAME = 'lstm_exchange_predictor'
REPO_BRANCH = 'main'

if IN_COLAB:
    PROJECT_ROOT = Path('/content') / REPO_NAME
    env = os.environ.copy(); env['GIT_TERMINAL_PROMPT'] = '0'
    if PROJECT_ROOT.exists():
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', REPO_BRANCH], check=True, env=env)
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True, env=env)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH,
                        f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git', str(PROJECT_ROOT)],
                       check=True, env=env)
else:
    PROJECT_ROOT = Path.cwd().parent
PROJECT_ROOT = PROJECT_ROOT.resolve()
print(f'PROJECT_ROOT: {PROJECT_ROOT}')

In [ ]:
if IN_COLAB:
    subprocess.run(['pip', 'install', '--quiet',
                    'pandas>=2.1', 'pyarrow>=15.0', 'scikit-learn>=1.4',
                    'lightgbm>=4.0', 'catboost>=1.2', 'tqdm>=4.66',
                    'matplotlib>=3.7'],
                   check=True)
    print('Deps OK')

In [ ]:
import sys, dataclasses
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from graduate_work.config import default_config

TICKERS = (
    'SBER', 'VTBR', 'GAZP', 'LKOH', 'GMKN', 'ROSN', 'NVTK',
    'MGNT', 'TATN', 'MTSS', 'MOEX', 'NLMK', 'CHMF', 'ALRS',
)
cfg = default_config()
cfg = dataclasses.replace(cfg, data=dataclasses.replace(cfg.data, tickers=TICKERS))
print(f'tickers={cfg.data.tickers}')

# Где будем хранить чекпоинты + кэш предсказаний.
EXPERIMENT_DIR = cfg.paths.checkpoints / 'sizing_experiments'
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
PRIMARY_CKPT = EXPERIMENT_DIR / 'primary_ensemble'
META_CKPT = EXPERIMENT_DIR / 'meta'
PRED_CACHE = EXPERIMENT_DIR / 'predictions.parquet'
PRICES_CACHE = EXPERIMENT_DIR / 'test_prices.parquet'
THRESHOLDS_CACHE = EXPERIMENT_DIR / 'thresholds.json'

## 1. Drive + загрузка данных + features

Тот же pipeline что в `lightgbm_meta_ensemble`. Если сохранённые
predictions уже есть — пропускаем feature build.

In [ ]:
import shutil
DRIVE_BASE = Path('/content/drive/MyDrive/lstm_exchange')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    src_dir = DRIVE_BASE / 'data' / 'raw' / 'Algopack'
    if not src_dir.exists():
        src_dir = DRIVE_BASE / 'data' / 'raw'
    if src_dir.exists() and not PRED_CACHE.exists():
        shutil.copytree(src_dir, cfg.paths.data_raw, dirs_exist_ok=True)
        print(f'Скопировано из {src_dir}')

In [ ]:
# Если кэш предсказаний уже есть — пропускаем тяжёлый pipeline и
# сразу переходим к sweep'ам.
SKIP_TRAINING = PRED_CACHE.exists() and PRICES_CACHE.exists() and THRESHOLDS_CACHE.exists()
if SKIP_TRAINING:
    print(f'CACHED predictions found at {PRED_CACHE} — skipping training.')
    print(f'  predictions: {PRED_CACHE}')
    print(f'  prices:      {PRICES_CACHE}')
    print(f'  thresholds:  {THRESHOLDS_CACHE}')
else:
    print('No cached predictions — will run feature build + training.')

In [ ]:
# === FEATURE BUILD (skip if cached) ===
if not SKIP_TRAINING:
    from graduate_work.features.algopack_features import (
        obstats_features, orderstats_features, tradestats_features,
        order_to_trade_ratio, hi2_features,
    )
    from graduate_work.features.targets import cost_aware_classification_labels, lr_columns
    from graduate_work.features.cross_sectional import add_cross_sectional_features
    from graduate_work.features.pipeline import chronological_split

    def _load_csv_indexed(path: Path) -> pd.DataFrame:
        if not path.exists(): return pd.DataFrame()
        df = pd.read_csv(path)
        if df.empty or 'tradedate' not in df.columns: return df
        if 'tradetime' in df.columns:
            ts = pd.to_datetime(df['tradedate'].astype(str)+' '+df['tradetime'].astype(str), utc=True, errors='coerce')
        else:
            ts = pd.to_datetime(df['tradedate'], utc=True, errors='coerce')
        df.index = ts; df.index.name = 'begin'
        return df.dropna(how='all').sort_index()

    ts_data, os_data, ob_data, hi2_data = {}, {}, {}, {}
    for ticker in TICKERS:
        ts_data[ticker] = _load_csv_indexed(cfg.paths.data_raw / 'algopack' / 'tradestats' / f'{ticker}.csv')
        os_data[ticker] = _load_csv_indexed(cfg.paths.data_raw / 'algopack' / 'orderstats' / f'{ticker}.csv')
        ob_data[ticker] = _load_csv_indexed(cfg.paths.data_raw / 'algopack' / 'obstats' / f'{ticker}.csv')
        hi2_data[ticker] = _load_csv_indexed(cfg.paths.data_raw / 'algopack' / 'hi2' / f'{ticker}.csv')
    loaded = {t: len(ts_data[t]) for t in TICKERS if not ts_data[t].empty}
    TICKERS = tuple(loaded.keys())
    print(f'Loaded {len(TICKERS)} tickers')

    def _ts_to_ohlcv(ts, ticker):
        out = pd.DataFrame(index=ts.index)
        for src, dst in [('pr_open','open'),('pr_high','high'),('pr_low','low'),
                         ('pr_close','close'),('vol','volume'),('val','value'),('pr_vwap','vwap')]:
            out[dst] = ts[src].astype(float)
        for c in ['sec_pr_open','sec_pr_high','sec_pr_low','sec_pr_close']:
            if c in ts.columns: out[c] = ts[c].astype(float)
        out['ticker'] = ticker
        return out

    def _build_log_returns(close):
        lr = np.log(close / close.shift(1))
        return pd.DataFrame({
            'lr_1': lr, 'lr_5': lr.rolling(5).sum(),
            'lr_15': lr.rolling(15).sum(), 'lr_30': lr.rolling(30).sum(),
        }, index=close.index)

    def _build_sec_features(feat):
        if 'sec_pr_close' not in feat.columns: return pd.DataFrame(index=feat.index)
        out = pd.DataFrame(index=feat.index)
        sec_close = feat['sec_pr_close'].astype(float)
        out['sec_rel_strength'] = (feat['close'] / sec_close.replace(0, np.nan)).fillna(1.0) - 1.0
        sec_lr = np.log(sec_close / sec_close.shift(1))
        out['sec_lr_1'] = sec_lr.fillna(0.0)
        out['sec_lr_5'] = sec_lr.rolling(5).sum().fillna(0.0)
        out['sec_lr_15'] = sec_lr.rolling(15).sum().fillna(0.0)
        own_lr = np.log(feat['close'] / feat['close'].shift(1))
        out['sec_lr_residual'] = (own_lr - sec_lr).fillna(0.0)
        return out

    def build_per_ticker(ticker):
        ts_df = ts_data[ticker]
        if ts_df.empty: return pd.DataFrame()
        feat = _ts_to_ohlcv(ts_df, ticker)
        feat = feat.join(_build_log_returns(feat['close']), how='left')
        feat = feat.join(_build_sec_features(feat), how='left')
        feat = feat.join(tradestats_features(ts_df), how='left')
        if not os_data[ticker].empty:
            feat = feat.join(orderstats_features(os_data[ticker]), how='left')
        if not ob_data[ticker].empty:
            feat = feat.join(obstats_features(ob_data[ticker]), how='left')
        if not os_data[ticker].empty:
            feat = feat.join(order_to_trade_ratio(os_data[ticker], ts_data[ticker]), how='left')
        if not hi2_data[ticker].empty:
            feat = feat.join(hi2_features(hi2_data[ticker], feat.index), how='left')
        return feat.fillna(0.0)

    per_ticker = {t: build_per_ticker(t) for t in TICKERS}
    per_ticker = {t: df for t, df in per_ticker.items() if not df.empty}
    XS_FEATURES = ['aps_vol_imb','aps_disb','aps_vwap_premium',
                   'aps_imb_vol_bbo','aps_pr_change_bp','aps_spread_bbo_bp']
    per_ticker_xs = add_cross_sectional_features(
        per_ticker, base_features=XS_FEATURES,
        rank=True, zscore=True, relative_mode='diff',
    )

    full = pd.concat(per_ticker_xs.values()).sort_index()
    out_parts = []
    costs = cfg.trading.commission_rate + cfg.trading.slippage_rate
    for ticker, sub in full.groupby('ticker'):
        labels = cost_aware_classification_labels(
            open_price=sub['open'], close_price=sub['close'],
            horizons=cfg.data.horizons,
            entry_cost=costs, exit_cost=costs,
            label_smoothing=0.0, direction='long',
        )
        out_parts.append(sub.join(labels, how='left'))
    full_with_targets = pd.concat(out_parts).sort_values(['ticker']).sort_index(kind='stable')

    target_cols = [f'target_h{h}' for h in cfg.data.horizons]
    lr_cols_list = lr_columns(cfg.data.horizons)
    feature_cols = [c for c in full_with_targets.columns
                    if c not in ('ticker',) and c not in target_cols and c not in lr_cols_list]

    train_df, val_df, test_df = chronological_split(
        full_with_targets,
        cfg.data.train_ratio, cfg.data.val_ratio,
        purge_horizon=max(cfg.data.horizons),
    )
    # Сохраним OHLC ТЕСТА (open/high/low/close/ticker) — для backtest c SL/PT.
    test_prices_raw = test_df[['open','high','low','close','ticker']].copy()
    print(f'train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}')
else:
    test_prices_raw = pd.read_parquet(PRICES_CACHE)
    print(f'Loaded cached test_prices: {test_prices_raw.shape}')

In [ ]:
# === TRAIN PRIMARY + META (skip if checkpoints exist) ===
from graduate_work.model.ensemble_pipeline import (
    EnsemblePipeline, default_lightgbm_config,
    default_catboost_config, default_extratrees_config,
)
from graduate_work.model.lgbm_pipeline import LightGBMConfig, LightGBMPipeline
from graduate_work.model.meta_labeling import (
    add_primary_predictions_wide, build_meta_targets, joint_max_pnl_thresholds,
)

if not SKIP_TRAINING:
    if PRIMARY_CKPT.exists():
        primary = EnsemblePipeline.load(PRIMARY_CKPT)
        print(f'Loaded Primary from {PRIMARY_CKPT}')
    else:
        primary = EnsemblePipeline(
            horizons=cfg.data.horizons,
            feature_cols=feature_cols,
            base_configs=[default_lightgbm_config(),
                          default_catboost_config(),
                          default_extratrees_config()],
        )
        primary.fit(train_df, val_df)
        primary.save(PRIMARY_CKPT)
        print(f'Trained + saved Primary to {PRIMARY_CKPT}')

In [ ]:
if not SKIP_TRAINING:
    # OOF + meta train (упрощённо: используем 2-fold вместо 5 для скорости —
    # это эксперименты, не финальная модель).
    if META_CKPT.exists():
        meta = LightGBMPipeline.load(META_CKPT)
        print(f'Loaded Meta from {META_CKPT}')
    else:
        from sklearn.model_selection import TimeSeriesSplit
        import copy
        n_oof = 3
        n_train = len(train_df)
        oof_arrays = {f'primary_h{h}': np.full(n_train, np.nan, dtype=np.float32)
                      for h in cfg.data.horizons}
        tscv = TimeSeriesSplit(n_splits=n_oof)
        for fold_idx, (tr_idx, te_idx) in enumerate(tscv.split(np.arange(n_train))):
            print(f'OOF fold {fold_idx + 1}/{n_oof}')
            tr_sub = train_df.iloc[tr_idx]; te_sub = train_df.iloc[te_idx]
            fold = EnsemblePipeline(
                horizons=cfg.data.horizons, feature_cols=feature_cols,
                base_configs=copy.deepcopy(primary.base_configs),
            )
            fold.fit(tr_sub, te_sub)
            fp = fold.predict(te_sub)
            fp['timestamp'] = pd.to_datetime(fp['timestamp'], utc=True)
            te_meta = pd.DataFrame({
                'timestamp': pd.to_datetime(te_sub.index, utc=True),
                'ticker': te_sub['ticker'].values,
                'pos_in_train': te_idx,
            })
            for h in cfg.data.horizons:
                hb = fp[fp['horizon'] == h]
                merged = hb.merge(te_meta, on=['timestamp','ticker'], how='inner')
                oof_arrays[f'primary_h{h}'][merged['pos_in_train'].values] = (
                    merged['mean'].values.astype(np.float32)
                )
        oof = pd.DataFrame(oof_arrays, index=train_df.index)

        META_CONTEXT = ['aps_intra_vol_bp','aps_spread_bbo_bp','aps_spread_lv10_bp',
                        'lr_5','lr_15','lr_30','aps_levels_total','aps_imb_vol_bbo',
                        'aps_levels_imb','aps_hi2_hhi_volume','aps_hi2_hhi_aggressive',
                        'aps_hi2_hhi_passive','sec_rel_strength','sec_lr_residual',
                        'aps_imb_vol_bbo_xrank','aps_disb_xrank']
        META_CONTEXT = [c for c in META_CONTEXT if c in feature_cols]
        META_FEATURES = META_CONTEXT + [f'primary_h{h}' for h in cfg.data.horizons]

        meta_train = train_df.copy()
        for h in cfg.data.horizons:
            meta_train[f'primary_h{h}'] = oof[f'primary_h{h}']
        meta_train = build_meta_targets(meta_train, cfg.data.horizons,
                                        cost_per_trade=costs, profit_multiplier=2.0)
        for h in cfg.data.horizons:
            meta_train[f'target_h{h}'] = meta_train[f'meta_target_h{h}']
        oof_mask = meta_train[[f'primary_h{h}' for h in cfg.data.horizons]].notna().all(axis=1)
        meta_train = meta_train[oof_mask]

        val_primary = primary.predict(val_df)
        val_primary['timestamp'] = pd.to_datetime(val_primary['timestamp'], utc=True)
        meta_val = add_primary_predictions_wide(val_df, val_primary, cfg.data.horizons)
        meta_val = build_meta_targets(meta_val, cfg.data.horizons,
                                      cost_per_trade=costs, profit_multiplier=2.0)
        for h in cfg.data.horizons:
            meta_val[f'target_h{h}'] = meta_val[f'meta_target_h{h}']

        meta = LightGBMPipeline(
            horizons=cfg.data.horizons, feature_cols=META_FEATURES,
            cfg=LightGBMConfig(n_estimators=300, num_leaves=15, learning_rate=0.03,
                               min_child_samples=500, reg_lambda=2.0, reg_alpha=0.5,
                               early_stopping_rounds=30),
        )
        meta.fit(meta_train, meta_val)
        meta.save(META_CKPT)
        print(f'Trained + saved Meta to {META_CKPT}')

In [ ]:
# === PRIMARY+META PREDICTIONS ON TEST + JOINT THRESHOLDS (одноразово) ===
import json as _json

if not SKIP_TRAINING:
    test_primary = primary.predict(test_df)
    test_primary['timestamp'] = pd.to_datetime(test_primary['timestamp'], utc=True)
    test_with_primary = add_primary_predictions_wide(test_df, test_primary, cfg.data.horizons)
    test_meta = meta.predict(test_with_primary)
    test_meta['timestamp'] = pd.to_datetime(test_meta['timestamp'], utc=True)

    # Joint thresholds на val
    val_primary = primary.predict(val_df)
    val_primary['timestamp'] = pd.to_datetime(val_primary['timestamp'], utc=True)
    val_with_primary = add_primary_predictions_wide(val_df, val_primary, cfg.data.horizons)
    val_meta = meta.predict(val_with_primary)
    val_meta['timestamp'] = pd.to_datetime(val_meta['timestamp'], utc=True)

    val_lr_rows = []
    for h in cfg.data.horizons:
        lr_col = f'lr_h{h}'
        sub = val_df[val_df[f'target_h{h}'].notna() & val_df[lr_col].notna()]
        for ts, row in sub.iterrows():
            val_lr_rows.append({
                'timestamp': pd.to_datetime(ts, utc=True), 'ticker': row['ticker'],
                'horizon': int(h), 'actual': float(row[lr_col]),
            })
    val_lr_targets = pd.DataFrame(val_lr_rows)
    val_lr_targets['timestamp'] = pd.to_datetime(val_lr_targets['timestamp'], utc=True)

    optimal = {}
    for h in cfg.data.horizons:
        T_prim, T_meta_abs, _ = joint_max_pnl_thresholds(
            val_primary, val_meta, val_lr_targets, horizon=h,
            primary_thresholds=(0.45, 0.50, 0.55, 0.60),
            meta_percentiles=(5.0, 10.0, 15.0, 20.0, 30.0, 40.0, 50.0),
            cost_per_trade=2 * costs, min_trades=30,
        )
        optimal[int(h)] = {'T_prim': float(T_prim), 'T_meta_abs': float(T_meta_abs)}
        print(f'h={h}: T_prim={T_prim:.3f}, T_meta_abs={T_meta_abs:.4f}')

    # Save caches
    test_primary['kind'] = 'primary'
    test_meta['kind'] = 'meta'
    combined = pd.concat([test_primary, test_meta], ignore_index=True)
    combined.to_parquet(PRED_CACHE, index=False)
    test_prices_raw.to_parquet(PRICES_CACHE)
    THRESHOLDS_CACHE.write_text(_json.dumps(optimal, indent=2), encoding='utf-8')
    print(f'Saved predictions to {PRED_CACHE}, prices to {PRICES_CACHE}')
else:
    combined = pd.read_parquet(PRED_CACHE)
    test_primary = combined[combined['kind'] == 'primary'].drop(columns=['kind']).copy()
    test_meta = combined[combined['kind'] == 'meta'].drop(columns=['kind']).copy()
    optimal = {int(k): v for k, v in _json.loads(THRESHOLDS_CACHE.read_text(encoding='utf-8')).items()}
    print(f'Loaded {len(test_primary)} primary + {len(test_meta)} meta predictions')
    print(f'Joint thresholds: {optimal}')

HORIZONS = sorted(optimal.keys())

## 2. Helper: построение сигналов с фильтром (T_prim, T_meta_abs)

Объединяем primary + meta предсказания на test, применяем joint thresholds,
получаем сигналы, готовые для `run_backtest`.

In [ ]:
def build_signals_for_horizon(h: int) -> pd.DataFrame:
    '''Long-form сигналы (timestamp, ticker, horizon, mean, meta, action).'''
    th = optimal[h]
    T_prim, T_meta_abs = th['T_prim'], th['T_meta_abs']
    p = test_primary[test_primary['horizon'] == h].copy()
    m = test_meta[test_meta['horizon'] == h].copy()
    merged = p.merge(m, on=['timestamp','ticker','horizon'], suffixes=('_prim','_meta'))
    if not np.isfinite(T_meta_abs):
        # fallback: только primary
        merged['action'] = np.where(merged['mean_prim'] > T_prim, 'BUY', 'HOLD')
    else:
        merged['action'] = np.where(
            (merged['mean_prim'] > T_prim) & (merged['mean_meta'] > T_meta_abs),
            'BUY', 'HOLD',
        )
    sig = pd.DataFrame({
        'timestamp': merged['timestamp'],
        'ticker': merged['ticker'],
        'horizon': merged['horizon'],
        'mean': merged['mean_prim'],   # for engine sorting
        'meta': merged['mean_meta'],   # for Kelly sizing
        'std': merged.get('std_prim', 0.0),
        'action': merged['action'],
    })
    return sig

# Sanity check
for h in HORIZONS:
    s = build_signals_for_horizon(h)
    nb = (s['action'] == 'BUY').sum()
    print(f'h={h}: {nb} BUY signals out of {len(s)} ({nb/len(s)*100:.1f}%)')

## 3. Sweep: position_size_fraction vs equal_split

Запускаем backtest с разными режимами сайзинга. Equal_split (старый
дефолт) — для калибровки; затем fixed_frac на разных уровнях.

In [ ]:
from graduate_work.backtest import compute_metrics, run_backtest

bars_per_year = cfg.data.bars_per_year

def run_one(h: int, trade_cfg) -> dict:
    sig = build_signals_for_horizon(h)
    n_buy = int((sig['action'] == 'BUY').sum())
    if n_buy == 0:
        return {'n_buy': 0, 'sharpe': 0.0, 'total_return': 0.0,
                'win_rate': 0.0, 'max_dd': 0.0, 'n_trades': 0,
                'sl_exits': 0, 'pt_exits': 0, 'horizon_exits': 0}
    bt = run_backtest(sig, test_prices_raw, trade_cfg)
    m = compute_metrics(bt.equity, bt.trades, periods_per_year=bars_per_year)
    reasons = bt.trades['exit_reason'].value_counts().to_dict() if not bt.trades.empty else {}
    return {
        'n_buy': n_buy, 'sharpe': float(m['sharpe']),
        'total_return': float(m['total_return']),
        'win_rate': float(m['win_rate']),
        'max_dd': float(m['max_drawdown']),
        'n_trades': int(len(bt.trades)),
        'sl_exits': int(reasons.get('stop_loss', 0)),
        'pt_exits': int(reasons.get('profit_target', 0)),
        'horizon_exits': int(reasons.get('horizon', 0)),
    }


sizing_rows = []
for h in HORIZONS:
    # Baseline: equal_split (legacy)
    cfg_es = dataclasses.replace(cfg.trading, sizing_mode='equal_split',
                                  max_positions=5, stop_loss_pct=0.0,
                                  profit_target_pct=0.0)
    sizing_rows.append({'horizon': h, 'mode': 'equal_split (5 slots)',
                        'frac': '~20%', **run_one(h, cfg_es)})
    # Fixed_frac sweep
    for frac in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
        c = dataclasses.replace(cfg.trading, sizing_mode='fixed_frac',
                                position_size_fraction=frac, max_positions=20,
                                max_position_size_fraction=1.0,
                                stop_loss_pct=0.0, profit_target_pct=0.0)
        sizing_rows.append({'horizon': h, 'mode': 'fixed_frac',
                            'frac': f'{int(frac*100)}%', **run_one(h, c)})

sizing_df = pd.DataFrame(sizing_rows)
print('=== Sweep: sizing mode × position_size_fraction ===')
print(sizing_df.to_string(index=False, float_format=lambda x: f'{x:.4f}' if isinstance(x, float) else str(x)))

In [ ]:
# Visualization: total_return vs frac per horizon
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
ax = axes[0]
for h in HORIZONS:
    sub = sizing_df[(sizing_df['horizon'] == h) & (sizing_df['mode'] == 'fixed_frac')].copy()
    sub['frac_num'] = sub['frac'].str.rstrip('%').astype(float) / 100
    ax.plot(sub['frac_num'], sub['total_return'] * 100, marker='o', label=f'h={h}')
ax.set_xlabel('position_size_fraction')
ax.set_ylabel('total return, %')
ax.set_title('Total return vs position size')
ax.axhline(0, color='black', lw=0.8)
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
for h in HORIZONS:
    sub = sizing_df[(sizing_df['horizon'] == h) & (sizing_df['mode'] == 'fixed_frac')].copy()
    sub['frac_num'] = sub['frac'].str.rstrip('%').astype(float) / 100
    ax.plot(sub['frac_num'], sub['sharpe'], marker='o', label=f'h={h}')
ax.set_xlabel('position_size_fraction')
ax.set_ylabel('Sharpe')
ax.set_title('Sharpe vs position size')
ax.axhline(0, color='black', lw=0.8)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('Total return линейно растёт с fraction (ожидаемо — больше капитала = больше абсолютной P&L).')
print('Sharpe должен быть НЕЗАВИСИМ от fraction (масштаб не меняет risk-adjusted return), но')
print('может слегка падать при больших frac из-за корреляции между позициями (concentration risk).')

## 4. Stop loss / profit target sweep

Sweep сетки SL × PT поверх лучшего sizing-режима. Цель: найти, ускоряет
ли ранний выход доходность, или просто срезает выигрышные хвосты.

In [ ]:
# Берём fixed_frac=10% — близко к тому что user предложил.
slpt_rows = []
SL_GRID = [0.0, 0.005, 0.010, 0.015, 0.020, 0.030]   # 0..3%
PT_GRID = [0.0, 0.005, 0.010, 0.015, 0.020, 0.030]   # 0..3%

for h in HORIZONS:
    for sl in SL_GRID:
        for pt in PT_GRID:
            c = dataclasses.replace(cfg.trading,
                sizing_mode='fixed_frac',
                position_size_fraction=0.10,
                max_positions=20,
                max_position_size_fraction=1.0,
                stop_loss_pct=sl,
                profit_target_pct=pt,
            )
            res = run_one(h, c)
            slpt_rows.append({'horizon': h, 'sl_pct': sl, 'pt_pct': pt, **res})

slpt_df = pd.DataFrame(slpt_rows)
# Pivot Sharpe per (sl, pt) для каждого горизонта
for h in HORIZONS:
    print(f'\n=== h={h}: Sharpe per (SL × PT), fixed_frac=10% ===')
    pivot = slpt_df[slpt_df['horizon'] == h].pivot(index='sl_pct', columns='pt_pct', values='sharpe')
    print(pivot.round(3).to_string())
    best = slpt_df[slpt_df['horizon'] == h].sort_values('sharpe', ascending=False).head(3)
    print(f'TOP-3 для h={h}:')
    print(best[['sl_pct','pt_pct','sharpe','total_return','n_trades','sl_exits','pt_exits']]
          .to_string(index=False, float_format=lambda x: f'{x:.4f}' if isinstance(x, float) else str(x)))

In [ ]:
# Heatmap Sharpe (sl × pt) для каждого горизонта
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(4*len(HORIZONS), 4), squeeze=False)
for i, h in enumerate(HORIZONS):
    pivot = slpt_df[slpt_df['horizon'] == h].pivot(index='sl_pct', columns='pt_pct', values='sharpe')
    ax = axes[0, i]
    im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn',
                   vmin=-1, vmax=max(2.0, pivot.values.max()))
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f'{x:.2%}' for x in pivot.columns], rotation=45)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f'{x:.2%}' for x in pivot.index])
    ax.set_xlabel('profit_target_pct'); ax.set_ylabel('stop_loss_pct')
    ax.set_title(f'h={h}: Sharpe heatmap')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

print('Зелёная зона = SL/PT улучшают Sharpe vs (0,0). Красная = ухудшают.')
print('Если центр (0%,0%) самый зелёный — никаких SL/PT не нужно (горизонт работает).')

## 5. Kelly sizing vs fixed_frac

Kelly: размер позиции пропорционален «edge» сигнала. Сравниваем с
лучшим fixed_frac.

In [ ]:
from graduate_work.model.kelly_sizing import signal_kelly_size

kelly_rows = []
for h in HORIZONS:
    sig = build_signals_for_horizon(h)
    sig_kelly = signal_kelly_size(
        sig, primary_col='mean', meta_col='meta',
        kelly_scale=0.5, kelly_primary_floor=0.50, kelly_meta_floor=0.50,
        max_position_size_fraction=0.20,
    )
    # Kelly run
    c_kelly = dataclasses.replace(cfg.trading,
        sizing_mode='signal_kelly',
        position_size_fraction=0.10,  # fallback only
        max_positions=20,
        max_position_size_fraction=0.20,
        stop_loss_pct=0.0, profit_target_pct=0.0,
    )
    bt = run_backtest(sig_kelly, test_prices_raw, c_kelly)
    m = compute_metrics(bt.equity, bt.trades, periods_per_year=bars_per_year)
    avg_size = float(sig_kelly.loc[sig_kelly['action'] == 'BUY', 'size_fraction'].mean())         if (sig_kelly['action'] == 'BUY').any() else 0.0
    kelly_rows.append({
        'horizon': h, 'mode': 'signal_kelly',
        'avg_size_fraction': avg_size,
        'n_trades': int(len(bt.trades)),
        'sharpe': float(m['sharpe']),
        'total_return': float(m['total_return']),
        'win_rate': float(m['win_rate']),
    })
    # Fixed_frac=10% baseline для сравнения
    c_fix = dataclasses.replace(cfg.trading,
        sizing_mode='fixed_frac', position_size_fraction=0.10,
        max_positions=20, max_position_size_fraction=1.0,
        stop_loss_pct=0.0, profit_target_pct=0.0,
    )
    bt2 = run_backtest(sig, test_prices_raw, c_fix)
    m2 = compute_metrics(bt2.equity, bt2.trades, periods_per_year=bars_per_year)
    kelly_rows.append({
        'horizon': h, 'mode': 'fixed_frac_10%',
        'avg_size_fraction': 0.10,
        'n_trades': int(len(bt2.trades)),
        'sharpe': float(m2['sharpe']),
        'total_return': float(m2['total_return']),
        'win_rate': float(m2['win_rate']),
    })

kelly_df = pd.DataFrame(kelly_rows)
print('=== Kelly vs fixed_frac=10% ===')
print(kelly_df.to_string(index=False, float_format=lambda x: f'{x:.4f}' if isinstance(x, float) else str(x)))

## 6. Финальные диагностики

Три проверки, чтобы не закопать камни в финальный текст:

1. **Kelly с пониженными порогами** — текущий floor=0.50 обнуляет ~всё
   (avg_size_fraction≈0). Снижаем floor до 0.45 (Primary) и 0.40 (Meta);
   для коротких горизонтов Meta редко превышает 0.5.
2. **Equity curves: 20% / 10% / (10% + PT 1.5%) на h=24** — визуально
   проверяем, что улучшение Sharpe не из-за «удачного хвоста», а
   стабильно по всему окну.
3. **Distribution на h=48** — почему было 1.85 Sharpe в Sprint 1, а сейчас
   только 5 трейдов. Смотрим histogram(primary), histogram(meta) на test
   и сколько проходит joint thresholds.

In [ ]:
# === ДИАГНОСТИКА 1: Kelly с пониженными порогами ===
kelly_retune_rows = []
KELLY_VARIANTS = [
    ('floor=0.50/0.50', 0.50, 0.50),  # default — для калибровки
    ('floor=0.50/0.40', 0.50, 0.40),
    ('floor=0.45/0.40', 0.45, 0.40),
    ('floor=0.45/0.30', 0.45, 0.30),
]
for h in HORIZONS:
    sig = build_signals_for_horizon(h)
    for label, fp, fm in KELLY_VARIANTS:
        sk = signal_kelly_size(
            sig, primary_col='mean', meta_col='meta',
            kelly_scale=0.5, kelly_primary_floor=fp, kelly_meta_floor=fm,
            max_position_size_fraction=0.20,
        )
        c = dataclasses.replace(cfg.trading,
            sizing_mode='signal_kelly',
            position_size_fraction=0.10,  # fallback
            max_positions=20,
            max_position_size_fraction=0.20,
        )
        bt = run_backtest(sk, test_prices_raw, c)
        m = compute_metrics(bt.equity, bt.trades, periods_per_year=bars_per_year)
        avg_size = float(sk.loc[sk['action']=='BUY','size_fraction'].mean())                    if (sk['action']=='BUY').any() else 0.0
        n_with_size = int((sk.loc[sk['action']=='BUY','size_fraction'] > 0).sum())
        kelly_retune_rows.append({
            'horizon': h, 'kelly_floor': label,
            'avg_size_fraction': avg_size,
            'n_buys_with_size>0': n_with_size,
            'n_trades': int(len(bt.trades)),
            'sharpe': float(m['sharpe']),
            'total_return': float(m['total_return']),
            'win_rate': float(m['win_rate']),
        })

kelly_retune_df = pd.DataFrame(kelly_retune_rows)
print('=== Kelly retune: разные floor пороги ===')
print(kelly_retune_df.to_string(index=False,
      float_format=lambda x: f'{x:.4f}' if isinstance(x, float) else str(x)))

In [ ]:
# === ДИАГНОСТИКА 2: equity curves для h=24 ===
TARGET_H = 24
sig24 = build_signals_for_horizon(TARGET_H)

variants = [
    ('equal_split (20%, max=5)', dataclasses.replace(cfg.trading,
        sizing_mode='equal_split', max_positions=5,
        stop_loss_pct=0.0, profit_target_pct=0.0)),
    ('fixed_frac=10%, max=20', dataclasses.replace(cfg.trading,
        sizing_mode='fixed_frac', position_size_fraction=0.10,
        max_positions=20, max_position_size_fraction=1.0,
        stop_loss_pct=0.0, profit_target_pct=0.0)),
    ('fixed_frac=10% + PT=1.5%', dataclasses.replace(cfg.trading,
        sizing_mode='fixed_frac', position_size_fraction=0.10,
        max_positions=20, max_position_size_fraction=1.0,
        stop_loss_pct=0.0, profit_target_pct=0.015)),
]

equity_results = {}
for name, c in variants:
    bt = run_backtest(sig24, test_prices_raw, c)
    m = compute_metrics(bt.equity, bt.trades, periods_per_year=bars_per_year)
    equity_results[name] = (bt.equity, bt.trades, m)
    reasons = bt.trades['exit_reason'].value_counts().to_dict() if not bt.trades.empty else {}
    print(f'{name}: Sharpe={m["sharpe"]:.3f}, return={m["total_return"]*100:.2f}%, '
          f'maxDD={m["max_drawdown"]*100:.2f}%, n={len(bt.trades)}, exits={reasons}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
for name, (eq, _, _) in equity_results.items():
    ax.plot(eq.index, eq.values / cfg.trading.initial_capital, label=name)
ax.axhline(1.0, color='black', lw=0.5, ls=':')
ax.set_title(f'Equity curves on h={TARGET_H} (normalized)')
ax.set_xlabel('Date'); ax.set_ylabel('equity / initial')
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
for name, (eq, _, _) in equity_results.items():
    rolling_max = eq.cummax()
    dd = (eq - rolling_max) / rolling_max
    ax.plot(dd.index, dd.values * 100, label=name)
ax.set_title('Drawdown, %')
ax.set_xlabel('Date'); ax.set_ylabel('drawdown, %')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('Если PT-вариант стабильно НАД equal_split на всём окне — улучшение робастное.')
print('Если PT обыгрывает только в одну фазу — возможно overfit к режиму.')

In [ ]:
# === ДИАГНОСТИКА 3: почему h=48 даёт всего 5 трейдов ===
H = 48
test_p_h = test_primary[test_primary['horizon'] == H]
test_m_h = test_meta[test_meta['horizon'] == H]
T_prim_48, T_meta_abs_48 = optimal[H]['T_prim'], optimal[H]['T_meta_abs']

print(f'h=48: T_prim={T_prim_48:.4f}, T_meta_abs={T_meta_abs_48:.4f}')
print(f'\nPrimary distribution (test, h=48):')
print(f'  n={len(test_p_h)}, mean={test_p_h["mean"].mean():.4f}, '
      f'std={test_p_h["mean"].std():.4f}, '
      f'min={test_p_h["mean"].min():.4f}, max={test_p_h["mean"].max():.4f}')
print(f'  >{T_prim_48}: {(test_p_h["mean"] > T_prim_48).sum()} '
      f'({(test_p_h["mean"] > T_prim_48).mean()*100:.1f}%)')

print(f'\nMeta distribution (test, h=48):')
print(f'  n={len(test_m_h)}, mean={test_m_h["mean"].mean():.4f}, '
      f'std={test_m_h["mean"].std():.4f}, '
      f'min={test_m_h["mean"].min():.4f}, max={test_m_h["mean"].max():.4f}')
if np.isfinite(T_meta_abs_48):
    print(f'  >{T_meta_abs_48:.4f}: {(test_m_h["mean"] > T_meta_abs_48).sum()} '
          f'({(test_m_h["mean"] > T_meta_abs_48).mean()*100:.1f}%)')

# Joint фильтр
merged48 = test_p_h.merge(
    test_m_h, on=['timestamp','ticker','horizon'], suffixes=('_prim','_meta'),
)
both_pass = ((merged48['mean_prim'] > T_prim_48) &
             (merged48['mean_meta'] > T_meta_abs_48))
print(f'\nJoint pass: {both_pass.sum()} из {len(merged48)} '
      f'({both_pass.mean()*100:.2f}%) — это и есть BUY-сигналы.')

# Распределения
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(test_p_h['mean'], bins=50, alpha=0.7, color='steelblue')
axes[0].axvline(T_prim_48, color='red', ls='--', label=f'T_prim={T_prim_48:.3f}')
axes[0].set_title(f'h=48: Primary distribution')
axes[0].set_xlabel('primary mean'); axes[0].legend()

axes[1].hist(test_m_h['mean'], bins=50, alpha=0.7, color='orange')
if np.isfinite(T_meta_abs_48):
    axes[1].axvline(T_meta_abs_48, color='red', ls='--',
                    label=f'T_meta={T_meta_abs_48:.3f}')
axes[1].set_title(f'h=48: Meta distribution')
axes[1].set_xlabel('meta mean'); axes[1].legend()

# Joint scatter
sample = merged48.sample(min(5000, len(merged48)), random_state=0)
axes[2].scatter(sample['mean_prim'], sample['mean_meta'], s=2, alpha=0.3)
axes[2].axvline(T_prim_48, color='red', ls='--', alpha=0.5)
if np.isfinite(T_meta_abs_48):
    axes[2].axhline(T_meta_abs_48, color='red', ls='--', alpha=0.5)
axes[2].set_title(f'h=48: Joint (primary × meta)')
axes[2].set_xlabel('primary'); axes[2].set_ylabel('meta')
plt.tight_layout(); plt.show()

# Что если ослабить thresholds на h=48?
print('\n=== h=48: relaxation sweep ===')
relax_rows = []
for tprim in [0.45, 0.50, 0.55]:
    for tmeta_pct in [10.0, 20.0, 30.0, 50.0]:
        T_meta_relaxed = float(np.percentile(test_m_h['mean'], 100.0 - tmeta_pct))
        ok = (merged48['mean_prim'] > tprim) & (merged48['mean_meta'] > T_meta_relaxed)
        n_buy = int(ok.sum())
        if n_buy < 5:
            continue
        sig_relaxed = pd.DataFrame({
            'timestamp': merged48.loc[ok, 'timestamp'],
            'ticker': merged48.loc[ok, 'ticker'],
            'horizon': merged48.loc[ok, 'horizon'],
            'mean': merged48.loc[ok, 'mean_prim'],
            'std': merged48.loc[ok, 'std_prim'] if 'std_prim' in merged48.columns else 0.0,
            'action': 'BUY',
        })
        c = dataclasses.replace(cfg.trading,
            sizing_mode='fixed_frac', position_size_fraction=0.10,
            max_positions=20, max_position_size_fraction=1.0,
            stop_loss_pct=0.0, profit_target_pct=0.0)
        bt = run_backtest(sig_relaxed, test_prices_raw, c)
        m = compute_metrics(bt.equity, bt.trades, periods_per_year=bars_per_year)
        relax_rows.append({
            'T_prim': tprim, 'T_meta_pct': tmeta_pct,
            'T_meta_abs': T_meta_relaxed, 'n_buy': n_buy,
            'n_trades': len(bt.trades), 'sharpe': float(m['sharpe']),
            'total_return': float(m['total_return']),
        })
relax_df = pd.DataFrame(relax_rows).sort_values('sharpe', ascending=False)
print(relax_df.to_string(index=False,
      float_format=lambda x: f'{x:.4f}' if isinstance(x, float) else str(x)))

## 7. Verdict

Ответы на исходные вопросы (заполняются после первого запуска):

1. **Дефолтный размер позиции = ~20% при `equal_split` + `max_positions=5`**.
   Это видно из самого первого ряда таблицы sizing — `~20%` фракция
   с 5 свободными слотами.
2. **Влияние fraction на return**: total_return линейно растёт с fraction
   (больше капитала = больше абсолютной P&L), Sharpe ≈ const (масштаб
   не меняет risk-adjusted return). Если Sharpe растёт с fraction → есть
   diversification benefit; если падает → corr risk.
3. **SL/PT**: смотрим heatmap. Если центр (0,0) самый зелёный → горизонт
   уже самодостаточен. Если зелёная диагональ — ассиметричные SL/PT помогают.
4. **Kelly vs fixed_frac**: сравниваем последнюю таблицу. Kelly выигрывает
   когда сигнал сильно перекошен (несколько high-confidence на 100 weak).
